In [ ]:
"""
    same plot as 4_fp_baseline
    but with 8 bit 
"""

# Autoreload
%load_ext autoreload
%autoreload 2

import gc
import ctypes
import csv
import json
import random
import subprocess
import sys
from pathlib import Path
from dataclasses import dataclass

import requests
from PIL import Image
import datasets
import numpy as np
import torch
from io import BytesIO
from matplotlib import pyplot as plt
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration


# load images
num_images = 5
FIXED_RESOLUTION = 448  # → 16×16 merged token grid (256 visual tokens)
seed = 42
_HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; research-bot/1.0)"}
VISUAL_LAYERS = [1, 2, 4, 8, 16, 24, 26, 27]  # matches latentlens-qwen2vl-embeddings index layers


def seed_all(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def gc_cuda():
    """Gargage collect RAM & Torch (CUDA) memory."""
    gc.collect()
    if torch.cuda.is_available():
        ctypes.CDLL("libc.so.6").malloc_trim(0)
        torch.cuda.empty_cache()

seed = 42
seed_all(seed)

def _ensure_nlp_resources():
    print("\nChecking NLP resources...")
    import nltk
    try:
        wn = __import__("nltk.corpus", fromlist=["wordnet"]).wordnet
        wn.synsets("dog")
    except Exception:
        print("  Downloading NLTK wordnet + omw-1.4...")
        nltk.download("wordnet", quiet=True)
        nltk.download("omw-1.4", quiet=True)

    try:
        import spacy
        spacy.load("en_core_web_sm")
    except OSError:
        print("  Downloading spaCy en_core_web_sm...")
        subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=True)

    print("  NLP resources OK")

_ensure_nlp_resources()


In [ ]:
# load val set, index, model

model_dir = Path("/workspace/latentlens/experiment/data/Qwen2-VL-7B-Instruct")
index_dir = Path("/workspace/latentlens/experiment/data/latentlens-qwen2vl-embeddings")
data_dir = Path('/workspace/latentlens/experiment/data/pixmo_cap_val100')

def load_data():
    ds_val100 = datasets.load_dataset(
        "parquet",
        data_files={"validation": str(data_dir / "data" / "validation-*.parquet")},
    )["validation"]

    print(f"Examples : {len(ds_val100)}")
    print(f"Columns  : {ds_val100.column_names}")
    print(f"val_idx  : {ds_val100[0]['val_idx']} … {ds_val100[-1]['val_idx']}")

    # Spot-check first example
    ex = ds_val100[0]
    img: Image.Image = ex["image"]
    print(f"\nExample 0")
    print(f"  val_idx   : {ex['val_idx']}")
    print(f"  image size: {img.size}  mode: {img.mode}")
    print(f"  image_url : {ex['image_url'][:80]}...")
    print(f"  caption   : {ex['caption'][:120].replace(chr(10), ' ')}...")
    print(f"  transcripts: {len(ex['transcripts'])} item(s)")

    # Quick grid preview of first 6 images
    fig, axes = plt.subplots(1, 6, figsize=(18, 3))
    for i, ax in enumerate(axes):
        ax.imshow(ds_val100[i]["image"])
        ax.set_title(f"val_idx={ds_val100[i]['val_idx']}", fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

    return ds_val100

@dataclass
class Bank:
    embeddings: torch.Tensor   # [N, hidden_dim] float16, kept on CPU
    metadata: list             # [{token_str, token_id, caption, position}, ...]


def load_banks(index_dir: Path, layers: list = None) -> dict:
    """Load reference embedding banks. Returns {layer: Bank}.
    If layers is None, loads all available layer_* directories."""
    if layers is None:
        layers = sorted(
            int(p.name.split("_")[1])
            for p in index_dir.iterdir()
            if p.is_dir() and p.name.startswith("layer_")
        )

    banks = {}
    for L in layers:
        path = index_dir / f"layer_{L}" / "embeddings_cache.pt"
        print(f"  Loading layer {L} from {path.name}...", end=" ", flush=True)
        data = torch.load(path, map_location="cpu", weights_only=False)
        banks[L] = Bank(embeddings=data["embeddings"], metadata=data["metadata"])
        e = data["embeddings"]
        print(f"shape={list(e.shape)} dtype={e.dtype}")

    return banks

def load_model(model_dir: Path):
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        str(model_dir), torch_dtype=torch.float16, device_map="auto"
    )

    model.eval()

    processor = AutoProcessor.from_pretrained(str(model_dir))
    px = FIXED_RESOLUTION * FIXED_RESOLUTION
    processor.image_processor.min_pixels = px
    processor.image_processor.max_pixels = px
    processor.image_processor.do_resize = False

    print(f"  Loaded model on {model.device}, resolution locked to {FIXED_RESOLUTION}×{FIXED_RESOLUTION}")
    return model, processor

data = load_data()
banks = load_banks(index_dir)
model, processor= load_model(model_dir)

In [ ]:
import gc
import sys
from tqdm import tqdm
from datasets import Dataset, Features, Sequence, Value

_EVAL_UTILS = Path("/workspace/latentlens/reproduce/scripts/evaluate")
if str(_EVAL_UTILS) not in sys.path:
    sys.path.insert(0, str(_EVAL_UTILS))
from utils import sample_valid_patch_positions

IMAGE_PAD_TOKEN_ID = 151655  # <|image_pad|> in Qwen2-VL vocabulary
PROMPT = "<|image_pad|>Describe this image."
GRID_SIZE = 16
BBOX_SIZE = 3
PROCESSED_SIZE = 512  # judge preprocessing canvas (center-crop, no padding)


def create_layer_wise_dataset(
    data: datasets.Dataset,
    model,
    processor,
    visual_layers: list[int] | None = None,
    seed: int = 42,
) -> datasets.Dataset:
    """Build layer-wise HF dataset: one row per (image, layer) with hidden state + patch coords.

    Patch selection matches evaluate_interpretability.py: seed once, then layers outer,
    images inner, sampling one valid 3x3 top-left position per (layer, image).
    """
    visual_layers = visual_layers or VISUAL_LAYERS
    image_mask = np.ones((PROCESSED_SIZE, PROCESSED_SIZE), dtype=bool)

    # Phase A: pre-compute patch assignments (CPU-only)
    random.seed(seed)
    assignments: dict[tuple[int, int], tuple[int, int]] = {}
    for layer in visual_layers:
        for example in data:
            val_idx = int(example["val_idx"])
            positions = sample_valid_patch_positions(
                image_mask,
                bbox_size=BBOX_SIZE,
                num_samples=1,
                grid_size=GRID_SIZE,
            )
            if not positions:
                raise RuntimeError(
                    f"No valid patch positions for val_idx={val_idx}, layer={layer}"
                )
            assignments[(layer, val_idx)] = positions[0]

    print(
        f"Pre-computed {len(assignments)} patch assignments "
        f"({len(data)} images x {len(visual_layers)} layers)"
    )

    # Phase B: one forward pass per image, extract hidden states for all layers
    rows: list[dict] = []
    for example in tqdm(data, desc="Extracting hidden states"):
        val_idx = int(example["val_idx"])
        img = example["image"].resize((FIXED_RESOLUTION, FIXED_RESOLUTION))
        inputs = processor(
            text=PROMPT,
            images=[img],
            return_tensors="pt",
        )
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)

        vision_positions = (inputs["input_ids"][0] == IMAGE_PAD_TOKEN_ID).nonzero(as_tuple=True)[0]
        if len(vision_positions) == 0:
            raise RuntimeError(f"No vision tokens found for val_idx={val_idx}")
        vision_start = int(vision_positions[0])

        for layer in visual_layers:
            if layer >= len(outputs.hidden_states):
                tqdm.write(
                    f"  Warning: layer={layer} out of range "
                    f"({len(outputs.hidden_states)} hidden states); skipping"
                )
                continue

            patch_row, patch_col = assignments[(layer, val_idx)]
            center_row = patch_row + BBOX_SIZE // 2
            center_col = patch_col + BBOX_SIZE // 2
            patch_token_idx = center_row * GRID_SIZE + center_col

            hs = (
                outputs.hidden_states[layer][0, vision_start + patch_token_idx, :]
                .float()
                .cpu()
                .numpy()
                .astype(np.float16)
            )

            rows.append(
                {
                    "val_idx": val_idx,
                    "layer": layer,
                    "patch_row": patch_row,
                    "patch_col": patch_col,
                    "center_row": center_row,
                    "center_col": center_col,
                    "patch_token_idx": patch_token_idx,
                    "hidden_state": hs,
                }
            )

        del outputs, inputs
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    features = Features(
        {
            "val_idx": Value("int32"),
            "layer": Value("int32"),
            "patch_row": Value("int32"),
            "patch_col": Value("int32"),
            "center_row": Value("int32"),
            "center_col": Value("int32"),
            "patch_token_idx": Value("int32"),
            "hidden_state": Sequence(Value("float16"), length=3584),
        }
    )
    ds = Dataset.from_list(rows, features=features)
    print(f"Created layer-wise dataset: {len(ds)} rows")
    return ds
